# Finetune the Twi TTS model

Continue training [`ghanaopendata/stable-twi-tts`](https://huggingface.co/ghanaopendata/stable-twi-tts) on your own audio —
to add a speaker, adapt to a domain, or improve a language it already knows.

**Finetuning is almost always the right starting point.** The published checkpoint has
already learned Twi phonetics, a 1,555-voice speaker table and a working vocoder. Starting
from it converges in hours where from-scratch takes days.

Use `train_from_scratch.ipynb` instead only if you are training a *different language* whose
phoneme inventory barely overlaps.


In [ ]:
# A GPU is required. On Colab: Runtime -> Change runtime type -> GPU.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# Piper's trainer, plus the two phonemisers. This takes a few minutes.
!apt-get -qq install -y espeak-ng > /dev/null
!git clone -q https://github.com/OHF-Voice/piper1-gpl.git
!pip install -q -e './piper1-gpl[train]'

# ghana-g2p for Twi. Installed from source because africa-g2p's wheel build is broken
# upstream (it ships data/ twice, so hatchling refuses the archive).
!git clone -q https://github.com/AfriSpeech/africa-g2p.git
!git clone -q https://github.com/GhanaNLP/ghana-g2p.git
import sys; sys.path[:0] = ['africa-g2p/src', 'ghana-g2p/src']
from ghana_g2p import GhanaG2P
print('twi g2p:', GhanaG2P('Asante Twi').ipa('Akwaaba', sep=' '))

In [ ]:
# VITS needs a Cython extension that pip does not build, and whose absence only shows up
# as ModuleNotFoundError once training starts. Build it now.
!pip install -q cython
%cd piper1-gpl/src
!python piper/train/vits/monotonic_align/setup.py build_ext --inplace 2>&1 | tail -2
# Upstream's __init__ imports from a nested path the build does not create.
!sed -i 's/^from \\.monotonic_align\\.core import/from .core import/' \
    piper/train/vits/monotonic_align/__init__.py
%cd /content
import torch
sys.path.insert(0, 'piper1-gpl/src')
from piper.train.vits import monotonic_align
print('monotonic_align ok', monotonic_align.maximum_path(
    torch.zeros(1, 3, 5), torch.ones(1, 3, 5)).shape)

In [ ]:
!git clone -q https://github.com/GhanaNLP/stable-twi-tts.git
import sys; sys.path.insert(0, 'stable-twi-tts')
!ls stable-twi-tts/training

## 1. Get the data

[`ghanaopendata/new-twi-tts-aligned-ipa`](https://huggingface.co/datasets/ghanaopendata/new-twi-tts-aligned-ipa) carries audio, text
and IPA together. It is ~23 GB, so on Colab mount Drive or use a subset first.


In [ ]:
from huggingface_hub import snapshot_download

# Start with a few shards to check the pipeline end to end before committing to 23 GB.
SHARDS = 4        # set to None for the whole dataset
repo = 'ghanaopendata/new-twi-tts-aligned-ipa'
patterns = [f'data/train-{i:05d}-*' for i in range(SHARDS)] if SHARDS else None
path = snapshot_download(repo, repo_type='dataset', local_dir='data',
                         allow_patterns=patterns)
!ls data/data | head -5 && du -sh data

## 2. Export wavs and a manifest

One shared wav directory at 22.05 kHz, plus a manifest. 22.05 kHz because the Piper
checkpoints we finetune from are 22.05 kHz — resampling once here beats fighting it later.


In [ ]:
!python stable-twi-tts/training/tts_data.py wavs \
    --data data/data --out tts22k --sr 22050 --threads 8
!head -2 tts22k/manifest.tsv | cut -c1-120

## 3. Phonemise from text

**This is the step that decides whether the model works.** Targets come from the same
phonemiser used at inference — ghana-g2p for Twi, espeak-ng for English — so there is no
train/inference gap by construction.


In [ ]:
!python stable-twi-tts/training/retarget_g2p.py \
    --data tts22k --piper-src piper1-gpl/src
!wc -l tts22k/metadata_train_g2p.csv tts22k/phonemes_g2p.json

## 4. Get the checkpoint and its id map

**The phoneme id map is not optional.** The weights encode *id 26 means /n/*. Load them
against a different map and the model reads a different language — with no error.


In [ ]:
from huggingface_hub import hf_hub_download
repo = 'ghanaopendata/stable-twi-tts'
ckpt = hf_hub_download(repo, 'finetune/checkpoint.ckpt')
cfg  = hf_hub_download(repo, 'finetune/config.json')
phon = hf_hub_download(repo, 'finetune/phonemes.json')
print(ckpt)
import json; c = json.load(open(cfg))
print('speakers', c['num_speakers'], '| symbols', len(c['phoneme_id_map']),
      '|', c['audio']['sample_rate'], 'Hz')

### Reuse the published id map

Keep every existing id fixed. If your data needs symbols the model lacks, add them in the
free slots (ids up to 255) and **graft their embeddings** from a pretrained model rather than
leaving them random — we measured that as the difference between converging in one epoch and
several.


In [ ]:
import shutil, json
shutil.copy(phon, 'tts22k/phonemes_g2p.json')   # published map wins

# Re-encode the metadata against it, so ids mean what the checkpoint thinks they mean.
!python stable-twi-tts/training/retarget_g2p.py \
    --data tts22k --old-map tts22k/phonemes_g2p.json --piper-src piper1-gpl/src
print('symbols:', len(json.load(open('tts22k/phonemes_g2p.json'))))

## 5. Match the speaker table

`emb_g` has one row per speaker, so a different speaker count fails to load. This resizes
it and seeds new rows from **real pretrained voices** rather than noise — a new speaker then
starts from a coherent voice and moves toward its target, instead of hunting for voice space.


In [ ]:
NSPK = !cut -d'|' -f2 tts22k/metadata_train_g2p.csv | sort -u | grep -c .
NSPK = int(NSPK[0]); print('speakers in your data:', NSPK)

!python stable-twi-tts/training/adapt_checkpoint.py \
    --ckpt {ckpt} --out adapted.ckpt --num-speakers {NSPK}

## 6. Clear the phoneme cache

Piper keys its cache on **text**, not on phoneme ids. If you reuse a cache directory after
changing the map, stale tensors are silently reused and you train on the old targets.


In [ ]:
!find runs -name '*.phonemes.pt' -delete 2>/dev/null; echo 'phoneme cache cleared'
!find runs -name '*.audio.pt' 2>/dev/null | wc -l   # audio cache kept: it is the slow one

## 7. Train

A finetune wants a **lower learning rate** than from-scratch — the default would undo what
the checkpoint knows. `--data.validation_split` is small on purpose: the default validates on
thousands of utterances through UTMOS, costing ~20% of throughput for a mean that stabilises
in a few hundred.


In [ ]:
!python -m piper.train fit \
    --data.voice_name my_twi_voice \
    --data.csv_path tts22k/metadata_train_g2p.csv \
    --data.audio_dir tts22k/wav \
    --data.dataset_type phoneme_ids \
    --data.phonemes_path tts22k/phonemes_g2p.json \
    --data.espeak_voice en-us \
    --data.phoneme_type text \
    --data.num_symbols 256 \
    --data.cache_dir runs/cache \
    --data.config_path runs/config.json \
    --data.batch_size 24 \
    --data.num_workers 4 \
    --data.validation_split 0.005 \
    --model.num_speakers {NSPK} \
    --model.sample_rate 22050 \
    --model.warmstart_ckpt adapted.ckpt \
    --model.mos_metric utmos \
    --optimizer.lr 5e-5 \
    --trainer.default_root_dir runs \
    --trainer.max_epochs -1 \
    --trainer.precision bf16-mixed \
    --trainer.val_check_interval 1000 \
    --trainer.accelerator gpu --trainer.devices 1

Two flags exist only to satisfy the CLI: `--data.espeak_voice` is required even in
`phoneme_ids` mode, where espeak is never called, and `--data.phoneme_type text` selects a
phonemiser that is never used but avoids constructing the espeak one, whose compiled bridge
a plain pip install does not build.


## 8. Pick a checkpoint by measurement

Not by the last step. Round-trip evaluation synthesises held-out phonemes, re-recognises
them and measures unit error — automatic, and unlike validation loss it tracks whether the
phonemes are actually being articulated.

**Read the gap against the real-audio floor, not the raw number.** Feeding real recordings
through gives the floor; the gap is what your model contributed.


In [ ]:
!ls runs/lightning_logs/version_0/checkpoints/

# best by perceptual quality (UTMOS) and by reconstruction, then compare by ear
import glob
best_mos = sorted(glob.glob('runs/**/epoch=*val_mos*.ckpt', recursive=True))[-1]
print('best val_mos:', best_mos)

In [ ]:
!python stable-twi-tts/training/synth.py \
    --checkpoint {best_mos} --config runs/config.json \
    --manifest tts22k/manifest_val.tsv --out synth --limit 100

!python stable-twi-tts/training/tts_eval.py \
    --manifest tts22k/manifest_val.tsv \
    --synth-dir synth --real-dir tts22k/wav --limit 100

## 9. Export a voice directory

Produces `model.onnx`, `config.json`, `voices.json`, `tokens.txt` and optionally a lexicon.
The ONNX export is not lossy — we measured 29.05% phoneme error against the PyTorch
checkpoint's 30.22% on identical input.


In [ ]:
!python stable-twi-tts/tools/export_voice.py \
    --checkpoint {best_mos} --train-config runs/config.json \
    --manifest tts22k/manifest.tsv --out voices/my_twi_voice \
    --top-n 10 --min-hours 0.5
!ls -la voices/my_twi_voice

### Rank voices by measurement, not by hours

`export_voice.py` ranks by training hours, which we found predicts almost nothing: our best
code-switch voice was 21st of 30 on pure Twi, and two of the three best Twi voices had under
3.3 h each. `tools/rank_voices.py` measures each voice instead.


In [ ]:
!python stable-twi-tts/tools/rank_voices.py \
    --voice-dir voices/my_twi_voice \
    --manifest tts22k/manifest_val.tsv \
    --synth-script stable-twi-tts/training/synth.py \
    --checkpoint {best_mos} --config runs/config.json \
    --asr-model ghananlpcommunity/ghana-speech-phoneme-asr \
    --per-voice 16

## Five things that fail silently

Each of these cost this project real time. None of them raise an error — they produce
fluent-sounding wrong audio, which is much more expensive than a crash.

1. **Phonemise with the same function at training and inference.** We trained on
   ASR-derived phonemes and synthesised from G2P-derived ones. They disagreed on 26% of
   units for Twi and 51% for English, and TTS phoneme error tracked it almost exactly
   (25.6% vs 68.6%). Nothing errored.
2. **Clear the phoneme cache when you change the id map.** Piper caches phoneme tensors
   keyed by *text*, not by ids. Change the map and the stale tensors are silently reused,
   so you train on the old targets believing you changed them. Delete `cache/*.phonemes.pt`
   and keep `*.audio.pt` — the audio cache is the expensive one.
3. **Never split IPA by character.** Units like `kʰ`, `t͡ʃ`, `k͡p`, `aɪ` are single symbols.
4. **Don't early-stop on `val_mel`.** It saturates while the adversarial losses are still
   removing artifacts. Keep top-k by `val_mos` too, and measure round-trip phoneme error.
5. **Resize the speaker table before loading a checkpoint with a different speaker count** —
   otherwise the load fails, or worse, silently ignores the speaker id.
